In [ ]:
# combining the analysis of all LipoGrid runs, first 2 plates are in positive mode only, next 2 are both positive and negative mode
import sys
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
from adjustText import adjust_text
import seaborn as sns

import scanpy as sc
import anndata as ad

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import mannwhitneyu

sys.path.append(str(Path.cwd().parent / "scripts"))
from style import set_default_style, set_publication_style
from volcano import plot_volcano, plot_volcano_lipid

# display / figure defaults
set_default_style(font_size=20)
%autosave 120

In [ ]:
# one aligned adata object from FOCUS pipeline with all 4 MALDI-MSI runs, first 2 in pos mode, 2 last ones in pos and neg mode
msi_int_cell = ad.read_h5ad('../data/lipogrid/pilot/analysis/final_4_runs/msi_int_cells.h5ad')
## split gRNA info into gene only
msi_int_cell.obs['gRNA_gene_only'] = msi_int_cell.obs['gRNA'].str.split('.').str[0]
msi_int_cell

In [ ]:
set_publication_style()

# Count cells per gRNA per sample_id
gRNA_sample_counts = (
    msi_int_cell.obs.groupby(['gRNA_gene_only', 'sample_id']).size().unstack(fill_value=0)
)
gRNA_sample_counts.columns = ['RUN01', 'RUN02', 'RUN03', 'RUN04']

# remove non-target categories
exclude = ['low_count', 'multiple_gRNAs', 'ambiguous', 'Intergenic', 'no_gRNA']
gRNA_sample_counts_filtered = gRNA_sample_counts.drop(index=exclude, errors='ignore')

# order by total count (descending)
gRNA_order = gRNA_sample_counts_filtered.sum(axis=1).sort_values(ascending=False).index
gRNA_sample_counts_ordered = gRNA_sample_counts_filtered.loc[gRNA_order]

colors = ["#601fb4", "#ff7f0e", "#28b2b4", "#d62799"]

# plot
fig, ax = plt.subplots(figsize=(12, 7), dpi=150)
gRNA_sample_counts_ordered.plot(kind='bar', stacked=True, color=colors, ax=ax,
                                width=0.6, edgecolor="none")

ax.set_xlabel('gRNA target gene')
ax.set_ylabel('Number of cells assigned')
ax.set_xticklabels([])                  # hide crowded gene labels
ax.tick_params(axis='x', length=0)
ax.tick_params(axis='y', length=5, width=1)
ax.legend(title='sample_id', frameon=False)

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/"
    "internalnorm_finalfigs/stackedbarplot_4runs_prefilter.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
set_publication_style()

# Data from your selection
counts = {
    'low count': 6733,
    'multi gRNAs': 15045,
 #   'intergenic': 473,
    'single gRNA': 57743,
}
labels = list(counts.keys())
values = list(counts.values())

fig, ax = plt.subplots(figsize=(4, 6), dpi=150)
ax.bar(labels, values, color=['#d62728', '#1f77b4', '#2ca02c'],
       width=0.8, edgecolor="none")

ax.set_ylabel('Number of cells')
ax.tick_params(axis='y', length=5, width=1)
ax.tick_params(axis='x', length=0)
plt.setp(ax.get_xticklabels(), rotation=60, ha='right')

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/internalnorm_finalfigs/barplot_gRNAdist_prefilter.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
set_publication_style()

df = msi_int_cell.var
offset = np.log2(1.8)

fig, axes = plt.subplots(2, 1, figsize=(7, 14), dpi=150)

for ax, mode in zip(axes, ["pos", "neg"]):
    sub = df[df["mz_mode"] == mode]
    x = np.log2(sub["avg_intensity_matrix"] + 1)   # matrix on x
    y = np.log2(sub["avg_intensity_cells"] + 1)    # cells on y

    ax.scatter(x, y, s=12, alpha=0.6, color="#005AB5", edgecolor="none")

    hi = max(x.max(), y.max())
    line = np.array([0, hi])

    ax.plot(line, line, "r:", lw=1)                              # diagonal
    ax.plot(line, line + offset, "g-", lw=1, label="cells = 1.8 × matrix")      # green line

    ax.set_xlim(0, hi)
    ax.set_ylim(0, hi)
    ax.set_aspect("equal")

    # same ticks on both axes
    ticks = MaxNLocator(nbins=8, integer=True).tick_values(0, hi)
    ticks = ticks[ticks <= hi]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(length=5, width=1)

    ax.set_xlabel("log₂(intensity matrix)")
    ax.set_ylabel("log₂(intensity cells)")
    ax.set_title(f"mz mode = {mode}")
    ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/"
    "internalnorm_finalfigs/scatterplot_mzint_prefilter.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
## keep only the mz values that are higher in the MSI profile of the cells compared to the matrix (folds calculated on the average profiles of the samples that have the measurements)
msi_int_cell_mzF = msi_int_cell[:, msi_int_cell.var['fold_change_cellVmat'] > 1.8].copy()
msi_int_cell_mzF.var['lipid_annotation_self'].value_counts() # 290 lipids remain after filtering on fold change > 1.8 of  over 10% are annotated)

## Split again in 2 for pos and neg mode to perform the final mz filtering

In [ ]:
msi_int_cell_mzF_pos = msi_int_cell_mzF[:, msi_int_cell_mzF.var['mz_mode'] == 'pos'].copy()
msi_int_cell_mzF_pos.obs['log2_total_counts'] = np.log2(msi_int_cell_mzF_pos.X.sum(axis=1)+1)

msi_int_cell_mzF_neg = msi_int_cell_mzF[:, msi_int_cell_mzF.var['mz_mode'] == 'neg'].copy()
msi_int_cell_mzF_neg = msi_int_cell_mzF_neg[~msi_int_cell_mzF_neg.obs['sample_id'].str.contains('RUN01')].copy()
msi_int_cell_mzF_neg.obs['log2_total_counts'] = np.log2(msi_int_cell_mzF_neg.X.sum(axis=1)+1)

msi_int_cell_mzF_pos, msi_int_cell_mzF_neg

In [ ]:
# calculate standard deviation of log2 total counts for pos and neg mode separately to set thresholds for filtering out low quality cells
print("Pos mode log2 total counts std:", msi_int_cell_mzF_pos.obs['log2_total_counts'].std())
print("Neg mode log2 total counts std:", msi_int_cell_mzF_neg.obs['log2_total_counts'].std())
## set thresholds based on the distribution of log2 total counts, to filter out low quality cells
## what 3 times the std below the mean is for pos and neg mode separately
pos_low_threshold = msi_int_cell_mzF_pos.obs['log2_total_counts'].mean() - 1.25 * msi_int_cell_mzF_pos.obs['log2_total_counts'].std()
pos_high_threshold = msi_int_cell_mzF_pos.obs['log2_total_counts'].mean() + 2.5 * msi_int_cell_mzF_pos.obs['log2_total_counts'].std()

neg_low_threshold = msi_int_cell_mzF_neg.obs['log2_total_counts'].mean() - 1.25 * msi_int_cell_mzF_neg.obs['log2_total_counts'].std()
neg_high_threshold = msi_int_cell_mzF_neg.obs['log2_total_counts'].mean() + 2.5 * msi_int_cell_mzF_neg.obs['log2_total_counts'].std()

print("Pos mode log2 total counts threshold:", pos_low_threshold, pos_high_threshold)
print("Neg mode log2 total counts threshold:", neg_low_threshold, neg_high_threshold)


In [ ]:
# plot histogram of the log1p_total_counts of the msi_intensities_per_cell_adf_2
# add vertical line at x = 7.5 in histogram
plt.figure(figsize=(10, 6))
plt.hist(msi_int_cell_mzF_pos.obs['log2_total_counts'], bins=100, color='blue', alpha=0.7)
plt.title('log2_total_counts pos mode')
plt.xlabel('log2_total_counts')
plt.ylabel('Frequency')
plt.axvline(x=pos_low_threshold, color='red', linestyle='--')  
plt.axvline(x=pos_high_threshold, color='red', linestyle='--') 

## negative mode has much less intensity, so each mode needs his own filters!
plt.figure(figsize=(10, 6))
plt.hist(msi_int_cell_mzF_neg.obs['log2_total_counts'], bins=100, color='blue', alpha=0.7)
plt.title('log2_total_counts neg mode')
plt.xlabel('log2_total_counts')
plt.ylabel('Frequency')
plt.axvline(x=neg_low_threshold, color='red', linestyle='--') 
plt.axvline(x=neg_high_threshold, color='red', linestyle='--')
plt.show()

In [ ]:
## filter cells based on  log2 total counts thresholds
msi_int_cell_mzF_pos = msi_int_cell_mzF_pos[(msi_int_cell_mzF_pos.obs['log2_total_counts'] > pos_low_threshold) & (msi_int_cell_mzF_pos.obs['log2_total_counts'] < pos_high_threshold)].copy()
msi_int_cell_mzF_neg = msi_int_cell_mzF_neg[(msi_int_cell_mzF_neg.obs['log2_total_counts'] > neg_low_threshold) & (msi_int_cell_mzF_neg.obs['log2_total_counts'] < neg_high_threshold)].copy()
msi_int_cell_mzF_pos, msi_int_cell_mzF_neg

In [ ]:
# Count cells per gRNA per sample_id
gRNA_sample_counts = msi_int_cell_mzF_pos.obs.groupby(['gRNA_gene_only', 'sample_id']).size().unstack(fill_value=0)
# change sample_id
gRNA_sample_counts.columns = ['RUN01', 'RUN02', 'RUN03', 'RUN04']
# remove 'low_count', 'multiple_gRNAs', 'ambiguous', 'Intergenic', 'no_gRNA' from gRNA_sample_counts
exclude = ['low_count', 'multiple_gRNAs', 'ambiguous', 'Intergenic', 'no_gRNA']
gRNA_sample_counts_filtered = gRNA_sample_counts.drop(index=exclude, errors='ignore')

# Order by total count (sum across all sample_id columns)
gRNA_order = gRNA_sample_counts_filtered.sum(axis=1).sort_values(ascending=False).index
gRNA_sample_counts_ordered = gRNA_sample_counts_filtered.loc[gRNA_order]

colors = ["#601fb4", '#ff7f0e', "#28b2b4", "#d62799"]  # 4 distinct colors

# Plot
gRNA_sample_counts_ordered.plot(kind='bar', stacked=True, figsize=(12,7), color=colors)
# plt.title('Stacked distribution of gRNA assignments across all runs', fontsize=20, fontweight='bold')
plt.xlabel('gRNA target gene', fontsize=24, fontweight='bold')
plt.ylabel('Number of cells assigned', fontsize=24, fontweight='bold')
plt.xticks(rotation=90, visible=False, fontsize=6, fontweight='bold')
plt.legend(title='sample_id')
plt.yticks(fontsize=22, fontweight='bold')
plt.legend(fontsize=22)
# plt.tight_layout()
plt.show()

In [ ]:
gRNA_gene_counts = msi_int_cell_mzF_pos.obs['gRNA_gene_only'].value_counts()
gRNA_gene_counts

In [ ]:
## drop gRNAs that are too lowly detected overall (likely lethal)
msi_int_cell_mzF_g_pos = msi_int_cell_mzF_pos[msi_int_cell_mzF_pos.obs['gRNA_gene_only'].isin(msi_int_cell_mzF_pos.obs['gRNA_gene_only'].value_counts()[msi_int_cell_mzF_pos.obs['gRNA_gene_only'].value_counts() >= 110].index)].copy()
msi_int_cell_mzF_g_neg = msi_int_cell_mzF_neg[msi_int_cell_mzF_neg.obs['gRNA_gene_only'].isin(msi_int_cell_mzF_pos.obs['gRNA_gene_only'].value_counts()[msi_int_cell_mzF_pos.obs['gRNA_gene_only'].value_counts() >= 110].index)].copy()
## drop multiple_gRNAs, low_count, no_gRNA, ambiguous
msi_int_cell_mzF_g_pos = msi_int_cell_mzF_g_pos[~msi_int_cell_mzF_g_pos.obs['gRNA_gene_only'].isin(['multiple_gRNAs', 'low_count', 'no_gRNA', 'ambiguous'])].copy()
msi_int_cell_mzF_g_neg = msi_int_cell_mzF_g_neg[~msi_int_cell_mzF_g_neg.obs['gRNA_gene_only'].isin(['multiple_gRNAs', 'low_count', 'no_gRNA', 'ambiguous'])].copy()
msi_int_cell_mzF_g_pos, msi_int_cell_mzF_g_neg

In [ ]:
## we have 4 LipoGrid runs each with their internal control cells (the intergenic targets), now first normalize the data per run to be able to compare the samples and do the analysis together, for this we will use the intergenic targets as internal controls for each run, and calculate fold changes of each cell compared to the average profile of the intergenic targets of the same run, then we can combine all runs together and do the analysis on the fold changes instead of the raw intensities, this should correct for any batch effects between the runs and make them comparable. We will do this separately for positive and negative mode data, as they have different intensity distributions and need different normalization. After normalization we can combine the data again and do dimensionality reduction and clustering to see if we can identify any patterns in the data related to the gRNA perturbations.

## First normalize data to Total Ion Chromatogram (TIC)
msi_int_cell_mzF_g_pos.obs['TIC'] = msi_int_cell_mzF_g_pos.X.sum(axis=1) 
average_TIC = round(msi_int_cell_mzF_g_pos.obs['TIC'].mean())
msi_int_cell_mzF_g_pos.obs['TIC_scaling_factor'] = msi_int_cell_mzF_g_pos.obs['TIC'] / average_TIC
msi_int_cell_mzF_g_pos_norm = msi_int_cell_mzF_g_pos.copy()
msi_int_cell_mzF_g_pos_norm.X = msi_int_cell_mzF_g_pos.X / msi_int_cell_mzF_g_pos.obs['TIC_scaling_factor'].values[:, np.newaxis]
## normalize data to Total Ion Chromatogram (TIC)
msi_int_cell_mzF_g_neg.obs['TIC'] = msi_int_cell_mzF_g_neg.X.sum(axis=1) 
average_TIC = round(msi_int_cell_mzF_g_neg.obs['TIC'].mean())
msi_int_cell_mzF_g_neg.obs['TIC_scaling_factor'] = msi_int_cell_mzF_g_neg.obs['TIC'] / average_TIC
msi_int_cell_mzF_g_neg_norm = msi_int_cell_mzF_g_neg.copy()
msi_int_cell_mzF_g_neg_norm.X = msi_int_cell_mzF_g_neg.X / msi_int_cell_mzF_g_neg.obs['TIC_scaling_factor'].values[:, np.newaxis]

## then calculate control profiles per run so per sample_id and mode
msi_int_cell_mzF_g_pos_norm.obs['is_control'] = msi_int_cell_mzF_g_pos_norm.obs['gRNA_gene_only'] == 'Intergenic'
msi_int_cell_mzF_g_neg_norm.obs['is_control'] = msi_int_cell_mzF_g_neg_norm.obs['gRNA_gene_only'] == 'Intergenic'

In [ ]:
def compute_control_profiles(adata, control_col='is_control', batch_col='sample_id'):
    """
    Compute mean expression profile of control cells per batch.
    Returns a DataFrame: rows = sample_id, columns = features
    """
    profiles = {}
    
    for sample_id, group in adata.obs.groupby(batch_col):
        # Get indices of control cells in this batch
        control_mask = group[control_col]
        control_idx = group.index[control_mask]
        
        if control_mask.sum() == 0:
            raise ValueError(f"No control cells found in sample {sample_id}")
        
        # Subset and compute mean (handle sparse matrix safely)
        X_control = adata[control_idx].X
        mean_profile = np.asarray(X_control.mean(axis=0)).squeeze()  # shape: (n_genes,)
        profiles[sample_id] = mean_profile
    
    return pd.DataFrame(profiles, index=adata.var_names).T  # shape: (n_samples, n_genes)


# Compute per-batch control profiles
control_profiles_pos = compute_control_profiles(msi_int_cell_mzF_g_pos_norm)
control_profiles_neg = compute_control_profiles(msi_int_cell_mzF_g_neg_norm)

# Mean control profile across all batches
mean_profile_pos = control_profiles_pos.mean(axis=0)  # shape: (n_genes,)
mean_profile_neg = control_profiles_neg.mean(axis=0)

# Correction factors per batch: how much to scale each batch to match the grand mean
# Add pseudocount to avoid division by zero on sparse m/z ratios and to prevent extreme correction factors for m/z values that are not detected in the control cells of a batch
pseudocount = 1

correction_factors_pos = ((mean_profile_pos + pseudocount) / (control_profiles_pos + pseudocount)).clip(lower=0.4, upper=2.5)  # limit correction factors to a reasonable range to avoid extreme scaling
correction_factors_neg = ((mean_profile_neg + pseudocount) / (control_profiles_neg + pseudocount)).clip(lower=0.4, upper=2.5)
# shape: (n_samples, n_genes) — one factor per gene per batch

In [ ]:
## histogram of correction factors for pos and neg mode
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.histplot(correction_factors_pos.values.flatten(), bins=50, kde=True, color='blue')
plt.title('Correction Factors - Positive Mode', fontsize=16, fontweight='bold')
plt.xlabel('Correction Factor', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')
plt.subplot(1, 2, 2)
sns.histplot(correction_factors_neg.values.flatten(), bins=50, kde=True, color='blue')
plt.title('Correction Factors - Negative Mode', fontsize=16, fontweight='bold')
plt.xlabel('Correction Factor', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
def apply_correction(adata, correction_factors, batch_col='sample_id'):
    """
    Apply per-batch, per-gene correction factors to all cells.
    Works on dense or sparse X — converts to dense for correction.
    """
    import scipy.sparse as sp
    
    X_corrected = adata.X.copy()
    if sp.issparse(X_corrected):
        X_corrected = X_corrected.toarray()
    
    for sample_id, group in adata.obs.groupby(batch_col):
        cell_idx = group.index
        factors = correction_factors.loc[sample_id].values  # shape: (n_genes,)
        
        # Multiply each cell in this batch by the correction vector
        row_idx = adata.obs_names.get_indexer(cell_idx)
        X_corrected[row_idx, :] *= factors
    
    adata_corrected = adata.copy()
    adata_corrected.X = X_corrected
    return adata_corrected


msi_pos_corrected = apply_correction(msi_int_cell_mzF_g_pos_norm, correction_factors_pos)
msi_neg_corrected = apply_correction(msi_int_cell_mzF_g_neg_norm, correction_factors_neg)

In [ ]:
# Intelligent filtering to keep mz values that appear due to a gene mutation in a subset of cells
# for each unique gRNA gene name calculate the mean intensity of each mz value
msi_pos_corrected.obs['gRNA_gene_only'] = msi_pos_corrected.obs['gRNA_gene_only'].astype('category')
mean_intensity_per_gRNA_pos = {}
for gRNA in msi_pos_corrected.obs['gRNA_gene_only'].cat.categories:
    mean_intensity_per_gRNA_pos[gRNA] = np.asarray(msi_pos_corrected[msi_pos_corrected.obs['gRNA_gene_only'] == gRNA].X.mean(axis=0)).flatten()
mean_intensity_per_gRNA_pos = pd.DataFrame(mean_intensity_per_gRNA_pos, index=msi_pos_corrected.var_names).T
# Check for each mz value if there is at least one gRNA with a mean intensity above 5
mean_intensity_per_gRNA_filter = (mean_intensity_per_gRNA_pos > 5).any(axis=0) # 1111 mz values
msi_int_cell_mzF_gRNAenr_pos = msi_pos_corrected[:, mean_intensity_per_gRNA_filter].copy()

msi_neg_corrected.obs['gRNA_gene_only'] = msi_neg_corrected.obs['gRNA_gene_only'].astype('category')
mean_intensity_per_gRNA_neg = {}
for gRNA in msi_neg_corrected.obs['gRNA_gene_only'].cat.categories:
    mean_intensity_per_gRNA_neg[gRNA] = np.asarray(msi_neg_corrected[msi_neg_corrected.obs['gRNA_gene_only'] == gRNA].X.mean(axis=0)).flatten()
mean_intensity_per_gRNA_neg = pd.DataFrame(mean_intensity_per_gRNA_neg, index=msi_neg_corrected.var_names).T

# Check for each mz value if there is at least one gRNA with a mean intensity above 5
mean_intensity_per_gRNA_filter = (mean_intensity_per_gRNA_neg > 5).any(axis=0) # 212 mz values
msi_int_cell_mzF_gRNAenr_neg = msi_neg_corrected[:, mean_intensity_per_gRNA_filter].copy()

msi_int_cell_mzF_gRNAenr_pos, msi_int_cell_mzF_gRNAenr_neg

In [ ]:
## histogram of highest value of mean intensity per gRNA for the mz values
## find highest value for each mz value across the gRNAs
max_mean_intensity_per_mz_pos = mean_intensity_per_gRNA_pos.max(axis=0).clip(upper=50) # clip values above 50 for better visualization
max_mean_intensity_per_mz_neg = mean_intensity_per_gRNA_neg.max(axis=0).clip(upper=50)

plt.figure(figsize=(10, 6))
plt.hist(max_mean_intensity_per_mz_pos, bins=100, color='blue', alpha=0.7)
plt.title('Max mean intensity per mz value across gRNAs (pos mode)')
plt.xlabel('Max mean intensity')
plt.ylabel('Frequency')
plt.axvline(x=5, color='red', linestyle='--')
plt.figure(figsize=(10, 6))
plt.hist(max_mean_intensity_per_mz_neg, bins=100, color='blue', alpha=0.7)
plt.title('Max mean intensity per mz value across gRNAs (neg mode)')
plt.xlabel('Max mean intensity')
plt.ylabel('Frequency')
plt.axvline(x=5, color='red', linestyle='--')
plt.show()

In [ ]:
len(msi_int_cell_mzF_gRNAenr_pos.var['lipid_annotation_self'].value_counts() ), len(msi_int_cell_mzF_gRNAenr_neg.var['lipid_annotation_self'].value_counts() ) # 62 lipids remain from 215 mzs

In [ ]:
# create UMAP
sc.pp.calculate_qc_metrics(msi_int_cell_mzF_gRNAenr_pos, inplace=True)
sc.pp.neighbors(msi_int_cell_mzF_gRNAenr_pos, n_neighbors=15, use_rep='X')
sc.tl.umap(msi_int_cell_mzF_gRNAenr_pos,  random_state=42)

# create UMAP keep correct color for the samples, so here need the 3rd and 4th normal colors green and red
sc.pp.neighbors(msi_int_cell_mzF_gRNAenr_neg, n_neighbors=15, use_rep='X')
sc.tl.umap(msi_int_cell_mzF_gRNAenr_neg,  random_state=42)

In [ ]:
SCANPY_4 = ['#1f77b4', '#ff7f0e', '#279e68', '#d62728']  # blue, orange, green, red
samples  = list(msi_int_cell_mzF_gRNAenr_pos.obs['sample_id'].cat.categories)
palette  = dict(zip(samples, SCANPY_4))
print(palette)

In [ ]:
set_publication_style()
sc.set_figure_params(dpi=150, frameon=True, vector_friendly=True, fontsize=12)

SAVE_DIR = "../data/lipogrid/pilot/analysis/internalnorm_finalfigs"


def styled_umap(adata, color, title, outname, **kwargs):
    """Draw a scanpy UMAP, apply the shared style, and save as an editable PDF."""
    ax = sc.pl.umap(
        adata, color=color, title=title, palette=palette,
        size=12, alpha=0.3, legend_fontsize=12, legend_fontweight='bold',
        wspace=0, show=False, **kwargs,
    )
    ax = ax[0] if isinstance(ax, (list, tuple)) else ax
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.tick_params(length=5, width=1)
    for spine in ax.spines.values():
        spine.set_linewidth(1)
    fig = ax.get_figure()
    fig.savefig(os.path.join(SAVE_DIR, outname), dpi=300, bbox_inches="tight")
    plt.show()


# pos / neg mode m/z UMAPs coloured by sample_id
styled_umap(msi_int_cell_mzF_gRNAenr_pos, "sample_id", "pos mode m/z values",
            "umap_sampleid_pos_prefilter.pdf")
styled_umap(msi_int_cell_mzF_gRNAenr_neg, "sample_id", "neg mode m/z values",
            "umap_sampleid_neg_prefilter.pdf")

In [ ]:
# for fair comparison, also plot UMAP on the same feature set but without the internal normalization
msi_int_cell_mzF_g_pos_norm_F = msi_int_cell_mzF_g_pos_norm[:, msi_int_cell_mzF_gRNAenr_pos.var_names].copy()
msi_int_cell_mzF_g_neg_norm_F = msi_int_cell_mzF_g_neg_norm[:, msi_int_cell_mzF_gRNAenr_neg.var_names].copy()
msi_int_cell_mzF_g_pos_norm_F, msi_int_cell_mzF_g_neg_norm_F

In [ ]:
sc.pp.neighbors(msi_int_cell_mzF_g_pos_norm_F, n_neighbors=15, use_rep='X')
sc.tl.umap(msi_int_cell_mzF_g_pos_norm_F,  random_state=42)

# create UMAP keep correct color for the samples, so here need the 3rd and 4th normal colors green and red
sc.pp.neighbors(msi_int_cell_mzF_g_neg_norm_F, n_neighbors=15, use_rep='X')
sc.tl.umap(msi_int_cell_mzF_g_neg_norm_F,  random_state=42)


In [ ]:
sc.pl.umap(msi_int_cell_mzF_g_pos_norm_F,  size=18, wspace=0, legend_fontsize=15, legend_fontweight='bold', color='sample_id', palette=palette , show=True , alpha=0.3, title='pos mode m/z values')
# keep only msi_int_cell_mzF_gRNAenr_pos_norm.obs['gRNA_gene_only'] = Intergenic and plot distribution in the UMAP
sc.pl.umap(msi_int_cell_mzF_g_neg_norm_F,  size=18, wspace=0, legend_fontsize=15, legend_fontweight='bold', color='sample_id' , palette=palette, show=True , alpha=0.3, title='neg mode m/z values')

In [ ]:
# group by gRNA_gene_only and take the mean of the intensities
msi_int_cell_mzF_gRNAenr_pos_norm_df = pd.DataFrame(msi_int_cell_mzF_gRNAenr_pos.X, index=msi_int_cell_mzF_gRNAenr_pos.obs['gRNA_gene_only'], columns=msi_int_cell_mzF_gRNAenr_pos.var_names)
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped = msi_int_cell_mzF_gRNAenr_pos_norm_df.groupby(msi_int_cell_mzF_gRNAenr_pos_norm_df.index).mean()
# log normalize the data
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_log = np.log2(msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped)
# devide through the Intergenic intensities to get the fold change, issue here is that you blow up sometimes the values because of the division!
# divide the intensities of all other rows by the intergenic row to get the fold change
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_fc = (msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped + 1)/(msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped.loc['Intergenic'] +1)
# log normalize the data
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_lfc = np.log2(msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_fc)
# remove the Intergenic row from the log fold change data
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_lfc = msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_lfc.drop('Intergenic', axis=0)  # drop the Intergenic row
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_lfc

# group by gRNA_gene_only and take the mean of the intensities
msi_int_cell_mzF_gRNAenr_neg_norm_df = pd.DataFrame(msi_int_cell_mzF_gRNAenr_neg.X, index=msi_int_cell_mzF_gRNAenr_neg.obs['gRNA_gene_only'], columns=msi_int_cell_mzF_gRNAenr_neg.var_names)
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped = msi_int_cell_mzF_gRNAenr_neg_norm_df.groupby(msi_int_cell_mzF_gRNAenr_neg_norm_df.index).mean()
# log normalize the data
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_log = np.log2(msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped)
# devide through the Intergenic intensities to get the fold change, issue here is that you blow up sometimes the values because of the division!
# divide the intensities of all other rows by the intergenic row to get the fold change
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_fc = (msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped + 1)/(msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped.loc['Intergenic'] +1)
# log normalize the data
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_lfc = np.log2(msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_fc)
# remove the Intergenic row from the log fold change data
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_lfc = msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_lfc.drop('Intergenic', axis=0)  # drop the Intergenic row
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_lfc

In [ ]:
## combine pos and negative intensities to plot UMAP
msi_int_cell_mzF_gRNAenr_norm_df_grouped = pd.concat([msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped, msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped], axis=1, join='outer')
## convert to anndata object
msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata = ad.AnnData(msi_int_cell_mzF_gRNAenr_norm_df_grouped, obs=msi_int_cell_mzF_gRNAenr_norm_df_grouped.index.to_frame(name='index'), var=msi_int_cell_mzF_gRNAenr_norm_df_grouped.columns.to_frame(name='mz'))
sc.pp.neighbors(msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata, n_neighbors=10, use_rep='X')
sc.tl.umap(msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata, min_dist=0.5, spread=1.0, random_state=42)
sc.pl.umap(msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata, size=18, wspace=0, legend_fontsize=15, legend_fontweight='bold' , show=True )

In [ ]:
set_publication_style()

new_dataset = pd.DataFrame(
    msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata.obsm['X_umap'],
    index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_anndata.obs.index,
    columns=['UMAP1', 'UMAP2'],
)

fig, ax = plt.subplots(figsize=(9, 9), dpi=150)
ax.scatter(new_dataset['UMAP1'], new_dataset['UMAP2'], s=0)  # invisible anchors

texts = []
for idx, row in new_dataset.iterrows():
    texts.append(
        ax.text(row['UMAP1'], row['UMAP2'], str(idx), fontsize=12, ha='center', va='center')
    )

ax.set_title('UMAP impact gene KO on m/z profile')
ax.set_xlabel('UMAP1')
ax.set_ylabel('UMAP2')
ax.tick_params(length=5, width=1)
ax.grid(False)

# Adjust text to avoid overlap
adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_text=(1.2, 1.2),
    expand_points=(1.2, 1.2),
    force_text=0.8,
    force_points=0.3,
    force_pull=20,
    time_lim=20,
    min_arrow_len=30,
)

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/"
    "internalnorm_finalfigs/UMAP_1377mz_impactKO.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
## combine pos and neg mode log fold change dataframes
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc = pd.concat([msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped_lfc, msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped_lfc], axis=1, join='outer')
## every value in adata_norm_grouped_lfc that is above 1 fold should be brought to 1 and below -1 to -1
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc[msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc > 1.5] = 1.5
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc[msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc < -1.5] = -1.5
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc

In [ ]:
# Set the number of clusters
n_clusters_cols = 7
# Fit AgglomerativeClustering to the transposed data (features as columns)
agglo = AgglomerativeClustering(n_clusters=n_clusters_cols, linkage='ward')
agglo_labels = agglo.fit_predict(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.T)

n_clusters_rows = 6
agglo_rows = AgglomerativeClustering(n_clusters=n_clusters_rows, linkage='ward')
agglo_labels_rows = agglo_rows.fit_predict(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc)

# Assign a color to each cluster

unique_clusters = np.unique(agglo_labels)
palette = sns.color_palette("Set2", len(unique_clusters))
cluster_colors = dict(zip(unique_clusters, palette))

unique_clusters_row = np.unique(agglo_labels_rows)
palette_row = sns.color_palette("Set2", len(unique_clusters_row))
cluster_colors_row = dict(zip(unique_clusters_row, palette_row))

# Map each column to its cluster color
col_colors = pd.Series(agglo_labels, index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.columns).map(cluster_colors)
row_colors = pd.Series(agglo_labels_rows, index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.index).map(cluster_colors_row)

# MANUAL CLUSTER ORDER
manual_order_col = [1,0,3,5,6,4,2]  # <-- change this to your desired order
manual_order_row = [3,5,1,4,0,2]  # <-- change this to your desired order

# Get column indices in the manual order
ordered_cols = []
for cl in manual_order_col:
    ordered_cols.extend(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.columns[agglo_labels == cl])

# Get row indices in the manual order
ordered_rows = []
for cl in manual_order_row:
    ordered_rows.extend(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.index[agglo_labels_rows == cl])


# Hierarchical clustering within each column cluster
ordered_cols_hier = []
for cl in manual_order_col:
    cols_in_cl = msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.columns[agglo_labels == cl]
    if len(cols_in_cl) > 1:
        # Hierarchical clustering for columns in this cluster
        Z = linkage(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc[cols_in_cl].T, method='average') # average
        idx = leaves_list(Z)
        ordered = cols_in_cl[idx]
    else:
        ordered = cols_in_cl
    ordered_cols_hier.extend(ordered)

# Use previous row ordering, or apply similar clustering for rows if needed
adata_norm_grouped_lfc_orderd_hier = msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.loc[ordered_rows, ordered_cols_hier]

g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r',
    figsize=(15, 5),
    col_cluster=False,
    row_cluster=False,
    cbar_kws={'label': 'log2FC'},
    xticklabels=False,
    yticklabels=False,
    col_colors=col_colors[ordered_cols_hier],
    row_colors=row_colors[ordered_rows]
)
g.ax_row_dendrogram.set_visible(False)
g.cax.set_position([.08, .3, .018, .25])
plt.show()

In [ ]:
# improve figure style and add intensity, enrichment and ion-mode bars
set_default_style(font_size=20)

# Concatenate positive and negative mode data
concatenated_expr = pd.concat([
    msi_int_cell_mzF_gRNAenr_pos.var['avg_intensity_cells'],
    msi_int_cell_mzF_gRNAenr_neg.var['avg_intensity_cells']
])
avg_expr = np.log2(concatenated_expr.loc[ordered_cols_hier]).clip(upper=6)

concatenated_foldmat = pd.concat([
    msi_int_cell_mzF_gRNAenr_pos.var['fold_change_cellVmat'],
    msi_int_cell_mzF_gRNAenr_neg.var['fold_change_cellVmat']
])
fold_overmatrix = np.log2(concatenated_foldmat.loc[ordered_cols_hier]).clip(upper=5)

# Create mode indicator: 0 for pos, 1 for neg
mode_indicator = pd.Series(
    [1]*len(msi_int_cell_mzF_gRNAenr_pos.var['avg_intensity_cells']) +
    [0]*len(msi_int_cell_mzF_gRNAenr_neg.var['avg_intensity_cells']),
    index=concatenated_expr.index
).loc[ordered_cols_hier]
mode_indicator_aligned = mode_indicator.loc[ordered_cols_hier]

fig = plt.figure(figsize=(40, 10))
gs = gridspec.GridSpec(4, 1, height_ratios=[7, 0.4, 0.4, 0.4])

# Main heatmap
ax0 = plt.subplot(gs[0])
hm0 = sns.heatmap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r',
    ax=ax0,
    cbar_kws={'pad': 0.03, 'label': 'log2FC over intergenic', 'orientation': 'vertical', 'shrink': 0.5, 'aspect': 12},
    xticklabels=False,
    yticklabels=False
)
ax0.set_xlabel('')
ax0.set_ylabel('')
hm0.collections[0].colorbar.ax.tick_params(labelsize=20)
hm0.collections[0].colorbar.set_label('log2FC over intergenic', fontsize=22)

# Average expression
ax1 = plt.subplot(gs[1], sharex=ax0)
hm1 = sns.heatmap(
    avg_expr.to_frame().T,
    cmap='plasma',
    ax=ax1,
    cbar_kws={'pad': 0.03, 'orientation': 'vertical', 'aspect': 1.5, 'ticks': [1, 5]},
    xticklabels=False,
    yticklabels=False
)
ax1.set_ylabel('')
ax1.set_xlabel('')
ax1.set_xticks([])
ax1.set_xticklabels([])
ax1.set_xlim(ax0.get_xlim())

# Enrichment
ax2 = plt.subplot(gs[2], sharex=ax0)
hm2 = sns.heatmap(
    fold_overmatrix.to_frame().T,
    cmap='viridis',
    ax=ax2,
    cbar_kws={'pad': 0.03, 'orientation': 'vertical', 'aspect': 1.5, 'ticks': [1, 4]},
    xticklabels=False,
    yticklabels=False
)
ax2.set_ylabel('')
ax2.set_xlabel('')
ax2.set_xlim(ax0.get_xlim())

# Mode indicator as the last (bottom) panel
ax_mode = plt.subplot(gs[3], sharex=ax0)
hm_mode = sns.heatmap(
    mode_indicator_aligned.to_frame().T,
    cmap=sns.color_palette(['#1f77b4', '#ff7f0e'], as_cmap=True),
    ax=ax_mode,
    cbar_kws={'pad': 0.03, 'orientation': 'vertical', 'aspect': 1.5},
    xticklabels=False,
    yticklabels=False
)
ax_mode.set_ylabel('')
ax_mode.set_xlabel('')
ax_mode.set_xticks([])
ax_mode.set_yticks([])
ax_mode.set_xlim(ax0.get_xlim())

plt.subplots_adjust(hspace=0.08)
plt.rcParams["pdf.fonttype"] = 42  # editable, embeddable text
plt.savefig("../data/lipogrid/pilot/analysis/internalnorm_finalfigs/heatmap_1377mz_impactKO.pdf", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
set_publication_style()

col_cluster = pd.Series(agglo_labels,
                        index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.columns)


def draw_cluster_boxes(ax, data, colors, manual_order, ylabel='',
                       showpoints=True, jitter=0.07):
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=False,
                    medianprops=dict(color='black', linewidth=2),
                    boxprops=dict(edgecolor='black', linewidth=1),
                    whiskerprops=dict(color='black', linewidth=1),
                    capprops=dict(color='black', linewidth=1))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
    if showpoints:
        for i, (vals, c) in enumerate(zip(data, colors), start=1):
            x = i + np.random.uniform(-jitter, jitter, size=len(vals))
            ax.scatter(x, vals, s=5, color=c, edgecolor='k', linewidth=0.2,
                       alpha=0.5, zorder=4)
    ax.axhline(0, color='grey', lw=1, ls='--')
    ax.set_xticks(range(1, len(manual_order) + 1))
    ax.set_xticklabels(range(1, len(manual_order) + 1))
    ax.set_ylabel(ylabel)
    ax.tick_params(length=5, width=1)


df = msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc
colors = [cluster_colors[cl] for cl in manual_order_col]


def drop_outliers(vals, k=3.0):
    """Remove points beyond k*IQR from the quartiles (Tukey 'far out' rule)."""
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) < 4:
        return vals
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return vals[(vals >= lo) & (vals <= hi)]


# build per-cluster data for each track
log2fc_data = [drop_outliers(df[col_cluster.index[col_cluster == cl]].mean(axis=1).values)
               for cl in manual_order_col]
expr_data   = [drop_outliers(avg_expr.loc[avg_expr.index.intersection(col_cluster.index[col_cluster == cl])].values)
               for cl in manual_order_col]
fold_data   = [drop_outliers(fold_overmatrix.loc[fold_overmatrix.index.intersection(col_cluster.index[col_cluster == cl])].values)
               for cl in manual_order_col]

fig, axes = plt.subplots(3, 1, figsize=(len(manual_order_col) * 1.0 + 2, 9),
                         sharex=True, dpi=150)
draw_cluster_boxes(axes[0], log2fc_data, colors, manual_order_col, ylabel='mean log₂FC')
draw_cluster_boxes(axes[1], expr_data,   colors, manual_order_col, ylabel='log₂ avg int')
draw_cluster_boxes(axes[2], fold_data,   colors, manual_order_col, ylabel='log₂ cell/matrix')
axes[2].set_xlabel('column cluster')

fig.tight_layout()
fig.savefig(
    '../data/lipogrid/pilot/analysis/'
    'final_4_runs/cluster_boxplots_per_mzcluster.pdf',
    bbox_inches='tight',
)
plt.show()

In [ ]:
# make anndata object with the grouped data
adata_norm_grouped_lfc_ad = ad.AnnData(X=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.values,
                                 obs=pd.DataFrame(index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.index),
                                 var=pd.DataFrame(index=msi_int_cell_mzF_gRNAenr_norm_df_grouped_lfc.columns)) 
adata_norm_grouped_lfc_ad

# check with only annotated lipids



In [ ]:
import re
# Join element 1 and 2, and then 4 and 5 if present, for each column name
# If the col name contains 'Chol xxx', change to 'Cholesterol'
def join_lipid_parts(col):
    if re.match(r'^Chol( |$)', col):
        return 'Cholesterol'
    parts = re.split(r' |(?:\+\;)', col)  # Split on space OR when '+;' occur together
    # Always join 1 and 2 (index 0 and 1)
    main = ' '.join(parts[:2])
    # Join 4 and 5 (index 3 and 4) if they exist
    extra = ' '.join(parts[3:5]) if len(parts) >= 5 else ''
    return main + ('/' + extra if extra else '')

In [ ]:
## update column names from index of msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped to msi_int_cell_mzF_gRNAenr_neg_norm.var['lipid_annotation_self']
msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped.columns = msi_int_cell_mzF_gRNAenr_pos.var['lipid_annotation_self'].values
msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped.columns = msi_int_cell_mzF_gRNAenr_neg.var['lipid_annotation_self'].values

msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname = pd.concat([
    msi_int_cell_mzF_gRNAenr_pos_norm_df_grouped,
    msi_int_cell_mzF_gRNAenr_neg_norm_df_grouped
], axis=1)
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname.columns.value_counts()

In [ ]:
## remove Unannotated columns 
msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname = msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname.loc[:, ~msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname.columns.str.contains('Unannotated')]

# also collapse same lipid with different ionizations
adata_norm_grouped_ad_lipidsonly_df_grouped = msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname.groupby(msi_int_cell_mzF_gRNAenr_norm_df_grouped_lipidname.columns.map(join_lipid_parts), axis=1).sum()

# Convert index to DataFrame for obs
obs_df = adata_norm_grouped_ad_lipidsonly_df_grouped.index.to_frame(index=False)

# Create AnnData object with grouped data and lipid annotations
adata_norm_grouped_ad_lipidsonly_grouped = ad.AnnData(
    X=adata_norm_grouped_ad_lipidsonly_df_grouped.values,
    obs=obs_df,
    var={'lipid_annotation_self': adata_norm_grouped_ad_lipidsonly_df_grouped.columns.values}
)

# Set var_names to lipid annotations as strings
adata_norm_grouped_ad_lipidsonly_grouped.obs_names = adata_norm_grouped_ad_lipidsonly_grouped.obs['gRNA_gene_only']
adata_norm_grouped_ad_lipidsonly_grouped.var_names = adata_norm_grouped_ad_lipidsonly_grouped.var['lipid_annotation_self'].astype(str)

# Show the AnnData object
adata_norm_grouped_ad_lipidsonly_grouped

In [ ]:
## merge lipid annotation to the lipid subclass and plot the main trends
adata_norm_grouped_ad_lipidsonly_grouped_collapsed = adata_norm_grouped_ad_lipidsonly_grouped.copy()
## remove columns with a / inside the var_names as they can be assigned to different lipids
adata_norm_grouped_ad_lipidsonly_grouped_collapsed = adata_norm_grouped_ad_lipidsonly_grouped_collapsed[:, ~adata_norm_grouped_ad_lipidsonly_grouped_collapsed.var_names.str.contains('/')]

adata_norm_grouped_ad_lipidsonly_grouped_collapsed.var['subclass'] = adata_norm_grouped_ad_lipidsonly_grouped_collapsed.var_names.str.split().str[0]
adata_norm_grouped_ad_lipidsonly_grouped_collapsed.var

# groupby subclass and take the mean
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass = adata_norm_grouped_ad_lipidsonly_grouped_collapsed.to_df().groupby(adata_norm_grouped_ad_lipidsonly_grouped_collapsed.var['subclass'], axis=1).sum()

## remove row with Intergenic
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.index = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.index.astype(str)
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.drop('Intergenic')

# remove columns with DG, PA and MG as they might be originating from larger lipid species
# adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.loc[:, ~adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.columns.str.contains('DG|PA|MG')]

## change Cholesterol to Chol in the column names
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.columns = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.columns.str.replace('Cholesterol', 'Chol')
 
# sum each column and keep the 15 highest columns
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_top_14 = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass[adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass.sum(axis=0).nlargest(18).index]
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_top_14

In [ ]:
set_publication_style()

## z transform the data to visualize the impact of gene KO on the lipid subclasses better
## clip to -4 and 4 to avoid outliers dominating the heatmap
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z = (
    adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_top_14
    - adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_top_14.mean()
) / adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_top_14.std()
adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z = \
    adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.clip(-4, 4)

# Set the number of clusters
n_clusters = 1
agglo = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
agglo_labels = agglo.fit_predict(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.T)

n_clusters_rows = 1
agglo_rows = AgglomerativeClustering(n_clusters=n_clusters_rows, linkage='ward')
agglo_labels_rows = agglo_rows.fit_predict(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z)

# Assign a color to each cluster
unique_clusters = np.unique(agglo_labels)
palette = sns.color_palette("Set2", len(unique_clusters))
cluster_colors = dict(zip(unique_clusters, palette))

unique_clusters_row = np.unique(agglo_labels_rows)
palette_row = sns.color_palette("Set2", len(unique_clusters_row))
cluster_colors_row = dict(zip(unique_clusters_row, palette_row))

# Map each column / row to its cluster color
col_colors = pd.Series(agglo_labels, index=adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.columns).map(cluster_colors)
row_colors = pd.Series(agglo_labels_rows, index=adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.index).map(cluster_colors_row)

# MANUAL CLUSTER ORDER
manual_order_col = [2, 1, 0]
manual_order_row = [2, 0, 1]

ordered_cols = []
for cl in manual_order_col:
    ordered_cols.extend(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.columns[agglo_labels == cl])

ordered_rows = []
for cl in manual_order_row:
    ordered_rows.extend(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.index[agglo_labels_rows == cl])

adata_norm_grouped_lfc_lipid_orderd2 = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.loc[ordered_rows, ordered_cols]

# Hierarchical clustering within each column cluster
ordered_cols_hier = []
for cl in manual_order_col:
    cols_in_cl = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.columns[agglo_labels == cl]
    if len(cols_in_cl) > 1:
        Z = linkage(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z[cols_in_cl].T, method='ward')
        ordered = cols_in_cl[leaves_list(Z)]
    else:
        ordered = cols_in_cl
    ordered_cols_hier.extend(ordered)

# Hierarchical clustering within each row cluster
ordered_rows_hier = []
for cl in manual_order_row:
    rows_in_cl = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.index[agglo_labels_rows == cl]
    if len(rows_in_cl) > 1:
        Z = linkage(adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.loc[rows_in_cl], method='ward')
        ordered = rows_in_cl[leaves_list(Z)]
    else:
        ordered = rows_in_cl
    ordered_rows_hier.extend(ordered)

adata_norm_grouped_lfc_orderd_hier = adata_norm_grouped_ad_lipidsonly_grouped_collapsed_subclass_z.loc[ordered_rows_hier, ordered_cols_hier]

# Plot heatmap with columns ordered by manual cluster order
g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier.T,
    cmap=sns.diverging_palette(250, 30, s=100, center="light", as_cmap=True),
    figsize=(16, 8),
    method='ward',
    metric='euclidean',
    col_cluster=False,
    row_cluster=True,
    cbar_kws={'label': 'z-score'},
    yticklabels=True,
    xticklabels=False,
    dendrogram_ratio=(0.1, 0.2),
    tree_kws={"linewidths": 1},   # dendrogram lines 1px, matching the rest
)

# apply shared styling to the clustermap parts
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
g.ax_heatmap.tick_params(length=5, width=1)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=14)
g.ax_heatmap.collections[0].colorbar.outline.set_linewidth(1)
g.cax.set_ylabel('z-score', fontsize=14, fontweight='bold')
g.cax.tick_params(length=5, width=1, labelsize=12)
g.cax.set_position([0.97, 0.40, 0.020, 0.2])  # [left, bottom, width, height]
mesh = g.ax_heatmap.collections[0]
mesh.set_edgecolor("face")
mesh.set_linewidth(0)

plt.rcParams["axes.grid"] = False
g.savefig(
    "../data/lipogrid/pilot/analysis/"
    "internalnorm_finalfigs/heatmap_lipidsubclass_ko.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
# devide through the Intergenic intensities to get the fold change, issue here is that you blow up sometimes the values because of the division!
# Check if fold changes are reasonable in MSI... ist quantitative but data doesnt go from 0 to 1... its 0 to 60 something most of the time.
# divide the intensities of all other rows by the intergenic row to get the fold change
adata_norm_grouped_fc_lipid = (adata_norm_grouped_ad_lipidsonly_df_grouped+1)/(adata_norm_grouped_ad_lipidsonly_df_grouped.loc['Intergenic'] +1)
# log normalize the data
adata_norm_grouped_lfc_lipid = np.log2(adata_norm_grouped_fc_lipid)
# remove the Intergenic row from the log fold change data
adata_norm_grouped_lfc_lipid = adata_norm_grouped_lfc_lipid.drop('Intergenic', axis=0)  # drop the Intergenic row
## every value in adata_norm_grouped_lfc that is above 1.5 fold should be brought to 1.5 and below -1.5 to -1.5
adata_norm_grouped_lfc_lipid[adata_norm_grouped_lfc_lipid > 1.5] = 1.5
adata_norm_grouped_lfc_lipid[adata_norm_grouped_lfc_lipid < -1.5] = -1.5

adata_norm_grouped_lfc_lipid

In [ ]:
# make anndata object with the grouped data
adata_norm_grouped_lfc_lipid_ad = ad.AnnData(X=adata_norm_grouped_lfc_lipid.values,
                                 obs=pd.DataFrame(index=adata_norm_grouped_lfc_lipid.index),
                                 var=pd.DataFrame(index=adata_norm_grouped_lfc_lipid.columns, data={'lipid_annotation': adata_norm_grouped_lfc_lipid.columns})) 
adata_norm_grouped_lfc_lipid_ad
sc.pp.neighbors(adata_norm_grouped_lfc_lipid_ad, n_neighbors=15, use_rep='X')
sc.tl.umap(adata_norm_grouped_lfc_lipid_ad,  random_state=42)
sc.pl.umap(adata_norm_grouped_lfc_lipid_ad,  size=18, wspace=0, legend_fontsize=15, legend_fontweight='bold', color='Cholesterol', show=True )


In [ ]:
new_dataset = pd.DataFrame(
	adata_norm_grouped_lfc_lipid_ad.obsm['X_umap'],
	index=adata_norm_grouped_lfc_lipid_ad.obs.index,
	columns=['UMAP1', 'UMAP2']
)

plt.figure(figsize=(9, 9))
# No need to plot dots if you only want text
texts = []
plt.scatter(new_dataset['UMAP1'], new_dataset['UMAP2'], s=0)  # Remove dots by setting s=0

for idx, row in new_dataset.iterrows():
    texts.append(
        plt.text(row['UMAP1'], row['UMAP2'], str(idx), fontsize=8, ha='center', va='center')
    )
plt.title('UMAP effect gene KO over intergenic controls')
plt.grid(True)
plt.tight_layout()

# Adjust text to avoid overlap
# adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=1))
adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_text=(1.2, 1.4),
    expand_points=(1.2, 1.4),
    force_text=0.3,
    force_points=0.3,
    force_pull=5,
    time_lim = 10,
    min_arrow_len=9,
)

plt.show()

In [ ]:
# Set the number of clusters
n_clusters = 5
# Fit Agglomerative Clustering to the transposed data (features as columns)
agglo = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
agglo_labels = agglo.fit_predict(adata_norm_grouped_lfc_lipid.T)

n_clusters_rows = 4
agglo_rows = AgglomerativeClustering(n_clusters=n_clusters_rows, linkage='ward')
agglo_labels_rows = agglo_rows.fit_predict(adata_norm_grouped_lfc_lipid)

# Assign a color to each cluster
import matplotlib.colors as mcolors

unique_clusters = np.unique(agglo_labels)
palette = sns.color_palette("Set2", len(unique_clusters))
cluster_colors = dict(zip(unique_clusters, palette))

unique_clusters_row = np.unique(agglo_labels_rows)
palette_row = sns.color_palette("Set2", len(unique_clusters_row))
cluster_colors_row = dict(zip(unique_clusters_row, palette_row))

# Map each column to its cluster color
col_colors = pd.Series(agglo_labels, index=adata_norm_grouped_lfc_lipid.columns).map(cluster_colors)
row_colors = pd.Series(agglo_labels_rows, index=adata_norm_grouped_lfc_lipid.index).map(cluster_colors_row)

# MANUAL CLUSTER ORDER
manual_order_col = [0,3,2,1,4]  # <-- change this to your desired order
manual_order_row = [0,2,1,3]  # <-- change this to your desired order

# Get column indices in the manual order
ordered_cols = []
for cl in manual_order_col:
    ordered_cols.extend(adata_norm_grouped_lfc_lipid.columns[agglo_labels == cl])

# Get row indices in the manual order
ordered_rows = []
for cl in manual_order_row:
    ordered_rows.extend(adata_norm_grouped_lfc_lipid.index[agglo_labels_rows == cl])

adata_norm_grouped_lfc_lipid_orderd2 = adata_norm_grouped_lfc_lipid.loc[ordered_rows, ordered_cols]

# Hierarchical clustering within each column cluster
ordered_cols_hier = []
for cl in manual_order_col:
    cols_in_cl = adata_norm_grouped_lfc_lipid.columns[agglo_labels == cl]
    if len(cols_in_cl) > 1:
        # Hierarchical clustering for columns in this cluster
        Z = linkage(adata_norm_grouped_lfc_lipid[cols_in_cl].T, method='average') # average
        idx = leaves_list(Z)
        ordered = cols_in_cl[idx]
    else:
        ordered = cols_in_cl
    ordered_cols_hier.extend(ordered)

# Use previous row ordering, or apply similar clustering for rows if needed
adata_norm_grouped_lfc_orderd_hier = adata_norm_grouped_lfc_lipid.loc[ordered_rows, ordered_cols_hier]

# Plot heatmap with columns ordered by manual cluster order
g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r',
    figsize=(28, 14),
    col_cluster=False,  # No hierarchical clustering
    row_cluster=False,
    cbar_kws={'label': 'log2FC'}, 
    yticklabels=False,
    xticklabels=True,
    col_colors=col_colors[ordered_cols],
    row_colors=row_colors[ordered_rows]
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=10)
g.cax.set_position([0.985, .50, .010, .15])  # [left, bottom, width, height] in figure coordinates
plt.tight_layout()
plt.show()

In [ ]:
# Plot heatmap with columns ordered by manual cluster order
g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r', figsize=(60, 20), col_cluster=False, row_cluster=False,
    cbar_kws={'label': 'log2FC'}, xticklabels=True, yticklabels=False,
)
# Set colorbar font size after creation
g.cax.tick_params(labelsize=40)
g.cax.set_ylabel('log2FC', fontsize=40)
# Adjust spacing in the plot
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=22)
g.ax_heatmap.tick_params(axis='x', labelrotation=90)  # Rotate x-axis labels for better spacing
g.ax_heatmap.tick_params(axis='y', labelsize=12)  # Adjust y-axis label size
g.ax_heatmap.set_title(f'Log2FC over intergenic controls {adata_norm_grouped_lfc_orderd_hier.shape[1]} lipid species', fontsize=70)
g.ax_heatmap.set_ylabel('')

g.cax.set_position([0.995, 0.35, 0.014, 0.25])  # [left, bottom, width, height] in figure coordinates
g.savefig('../data/lipogrid/pilot/analysis/final_4_runs/heatmap_log2FC_158lipid_noIons_internalnorm_annotation_143genes_small.pdf')
plt.show()


In [ ]:
set_publication_style()

# gene (row) and lipid (column) selections
genes_to_label = ['GPAM','PNPLA2','LPCAT3','ACOX1','ACOT4','PTDSS1','NPC1','DECR1','HMGCR','FADS2','ANGPTL4','MBOAT7','SCD','ELOVL6']
terms_to_label = ['SM 42:2;O2','DG 30:0','MG 20:4','MG 18:1','SM 34:1;O2','Cholesterol','PC O-34:1/PC P-34:0',
 'Cer 34:1;O2','Hex2Cer 34:2;O2','CAR 20:2','LPC 22:5','LPI 18:1','PG 44:12','LPE 22:6','PA 40:9',
 'LPA 22:4','PE 34:0','LPE 18:1','PS 44:8','PE 40:4','PC 32:0','PA 42:6','LPI 20:4','PA 36:2',
 'PI 36:2','PA 38:5','LPG 16:0','PS 36:1','LPC 16:1','PE 36:5']
term_rename = {}

# subset the matrix, keeping the existing (hierarchical) order
df = adata_norm_grouped_lfc_orderd_hier
miss_g = [x for x in genes_to_label if x not in df.index]
miss_t = [x for x in terms_to_label if x not in df.columns]
if miss_g: print(f"[labels] genes not in matrix: {sorted(miss_g)}")
if miss_t: print(f"[labels] terms not in matrix: {sorted(miss_t)}")
rows_keep = [g for g in df.index   if g in set(genes_to_label)]   # preserves df order
cols_keep = [c for c in df.columns if c in set(terms_to_label)]
sub = df.loc[rows_keep, cols_keep]

# compact clustermap
nrow, ncol = sub.shape
g = sns.clustermap(
    sub, cmap='RdYlBu_r',
    figsize=(0.25*ncol + 2, 0.25*nrow + 2),
    col_cluster=False, row_cluster=False,
    cbar_kws={'label': 'log2FC'}, xticklabels=True, yticklabels=True,
    dendrogram_ratio=(0.01, 0.01), linewidths=0,
)

g.ax_heatmap.set_xticklabels([term_rename.get(t.get_text(), t.get_text())
                              for t in g.ax_heatmap.get_xticklabels()],
                             rotation=90, fontsize=11)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=12)
g.ax_heatmap.tick_params(length=0)
g.ax_heatmap.set_ylabel('')
g.ax_heatmap.set_title(f'Log2FC over intergenic controls — {ncol} selected lipid species', fontsize=14, pad=16)
for s in g.ax_heatmap.spines.values():
    s.set_visible(True); s.set_linewidth(1)

# colorbar styling
g.cax.set_ylabel('log2FC', fontsize=12, fontweight='bold')
g.cax.tick_params(labelsize=11, length=5, width=1)
g.cax.set_position([1.0, 0.35, 0.02, 0.3])

g.savefig('../data/lipogrid/pilot/analysis/internalnorm_finalfigs/heatmap_log2FC_selected_compact.pdf', bbox_inches='tight')
plt.show()

## Perform basic statistics on individual gene KO's

In [ ]:
# pre-process and sum same lipids, here as the number of cells is different for pos and neg mode, will need to do the statistics sepertate and then concatenate
# also remove the m/z values that are not annotated
msi_int_cell_mzF_gRNAenr_pos_norm_df2 = msi_int_cell_mzF_gRNAenr_pos.to_df()
msi_int_cell_mzF_gRNAenr_neg_norm_df2 = msi_int_cell_mzF_gRNAenr_neg.to_df()
msi_int_cell_mzF_gRNAenr_pos_norm_df2.index = msi_int_cell_mzF_gRNAenr_pos.obs['gRNA_gene_only']  
msi_int_cell_mzF_gRNAenr_neg_norm_df2.index = msi_int_cell_mzF_gRNAenr_neg.obs['gRNA_gene_only']  
msi_int_cell_mzF_gRNAenr_pos_norm_df2.columns = msi_int_cell_mzF_gRNAenr_pos.var['lipid_annotation_self'].astype(str)
msi_int_cell_mzF_gRNAenr_neg_norm_df2.columns = msi_int_cell_mzF_gRNAenr_neg.var['lipid_annotation_self'].astype(str)

# remove the 'No_match_15ppm' column
msi_int_cell_mzF_gRNAenr_pos_norm_df2 = msi_int_cell_mzF_gRNAenr_pos_norm_df2.loc[:, msi_int_cell_mzF_gRNAenr_pos_norm_df2.columns != 'Unannotated']  
# also collapse same lipid with different ionizations
msi_int_cell_mzF_gRNAenr_pos_norm_df2_grouped = msi_int_cell_mzF_gRNAenr_pos_norm_df2.groupby(msi_int_cell_mzF_gRNAenr_pos_norm_df2.columns.map(join_lipid_parts), axis=1).sum()

## clean up dataset by removing gRNA genes with less than 100 cells and is not multiple_gRNAs or low_count
gRNAs_to_keep = gRNA_gene_counts[gRNA_gene_counts > 100 ] # keep gRNAs with more than 100 cells, more lenient as we are performing statistics anyways
# remove gRNAs  multiple_gRNAs, low_count and Intergenic
gRNAs_to_keep = gRNAs_to_keep[~gRNAs_to_keep.index.isin(['multiple_gRNAs', 'low_count', 'no_gRNA', 'ambiguous','PSMB3', 'SNRNP200'])]
msi_int_cell_mzF_gRNAenr_pos_norm_df2_grouped = msi_int_cell_mzF_gRNAenr_pos_norm_df2_grouped.loc[gRNAs_to_keep.index]

msi_int_cell_mzF_gRNAenr_neg_norm_df2 = msi_int_cell_mzF_gRNAenr_neg_norm_df2.loc[:, msi_int_cell_mzF_gRNAenr_neg_norm_df2.columns != 'Unannotated']  
msi_int_cell_mzF_gRNAenr_neg_norm_df2_grouped = msi_int_cell_mzF_gRNAenr_neg_norm_df2.groupby(msi_int_cell_mzF_gRNAenr_neg_norm_df2.columns.map(join_lipid_parts), axis=1).sum()
msi_int_cell_mzF_gRNAenr_neg_norm_df2_grouped = msi_int_cell_mzF_gRNAenr_neg_norm_df2_grouped.loc[gRNAs_to_keep.index]
msi_int_cell_mzF_gRNAenr_neg_norm_df2_grouped

In [ ]:
## calculate statistics for each gene KO over intergenic controls for all lipid species that made the
## cutoff, generating a results table with log2FC and Mann-Whitney p-value (run once per ion mode)
def compute_ko_lipid_stats(df_grouped, out_csv):
    results_df = pd.DataFrame(index=df_grouped.columns)  # lipids as index
    intergenic_gRNAs = df_grouped[df_grouped.index.str.contains('Intergenic')]

    for target in df_grouped.index.unique():
        target_gRNAs = df_grouped[df_grouped.index.str.match(target)]

        target_gRNAs_mean = target_gRNAs.mean(axis=0)
        intergenic_gRNAs_mean = intergenic_gRNAs.mean(axis=0)
        target_gRNAs_lfc = np.log2((target_gRNAs_mean + 1) / (intergenic_gRNAs_mean + 1))

        p_values = [
            mannwhitneyu(target_gRNAs[col], intergenic_gRNAs[col], alternative='two-sided')[1]
            for col in target_gRNAs.columns
        ]

        results_df[f'{target}_log2FC'] = target_gRNAs_lfc.values
        results_df[f'{target}_pvalue'] = p_values

    results_df.reset_index(inplace=True)
    results_df.set_index(results_df['lipid_annotation_self'], inplace=True)
    results_df = results_df.drop(columns=['lipid_annotation_self'])
    results_df.to_csv(out_csv)
    return results_df


all_results_pos_df = compute_ko_lipid_stats(
    msi_int_cell_mzF_gRNAenr_pos_norm_df2_grouped,
    '../data/lipogrid/pilot/analysis/final_4_runs/all_lipid_pos_log2FC_pvalues_per_gene.csv',
)
all_results_pos_df

In [ ]:
all_results_neg_df = compute_ko_lipid_stats(
    msi_int_cell_mzF_gRNAenr_neg_norm_df2_grouped,
    '../data/lipogrid/pilot/analysis/final_4_runs/all_lipid_neg_log2FC_pvalues_per_gene.csv',
)
all_results_neg_df

In [ ]:
# Concatenate the two DataFrames, as the intensity scales between positive and negative ion mode is too different to simply sum lipid species that are found in both modes, we will perform the statistics seperate for each mode and then select the mode where this lipid specie was best detected.
all_results_concat = pd.concat([all_results_pos_df, all_results_neg_df], axis=0)

# Compute the median p-value across all columns containing '_pvalue'
all_results_concat['medianP'] = all_results_concat[[col for col in all_results_concat.columns if '_pvalue' in col]].median(axis=1)

# Add lipid identifier as a column if not already present
all_results_concat['lipid'] = all_results_concat.index

# Sort by median p-value (most significant first)
all_results_concat_sorted = all_results_concat.sort_values('medianP', ascending=True)

# Drop duplicates, keeping the row with the lowest median p-value for each lipid
all_results_df = all_results_concat_sorted.drop_duplicates(subset='lipid', keep='first')

# Show the final DataFrame
all_results_df.to_csv('../data/lipogrid/pilot/analysis/final_4_runs/all_lipid_log2FC_pvalues_per_gene.csv')

all_results_df

In [ ]:
plot_volcano(all_results_df, target='NPC1')

In [ ]:
plot_volcano_lipid(all_results_df, lipid='Cholesterol')

In [ ]:
plot_volcano(all_results_df, target='MBOAT7')

In [ ]:
## boxplots of substrate selections
# optional: path to HelveticaNeue.ttf (leave as None to use default sans-serif)
FONT_PATH = None  # e.g. ".../HelveticaNeue_ttf/HelveticaNeue.ttf"

# alternate green/purple palette (distinct from the ACSL figures)
GREEN_FILL, PURPLE_FILL = "#7FBF6B", "#B06FD6"
GREEN_PT,   PURPLE_PT   = "#4E9E3E", "#8A3FBF"

# font / style
fam = None
if FONT_PATH:
    fm.fontManager.addfont(FONT_PATH)
    fam = fm.FontProperties(fname=FONT_PATH).get_name()
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ([fam] if fam else []) + ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 12,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
})

def ps_db(ann):
    """Double bonds of a diacyl PS species; None if not eligible."""
    ann = ann.strip()
    if "/" in ann or ";" in ann:
        return None
    m = re.match(r"^PS\s+(O-|P-)?(\d+):(\d+)$", ann)
    if not m or m.group(1):     # exclude ether/plasmalogen
        return None
    return int(m.group(3))


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


# collect data
df = all_results_df.apply(pd.to_numeric, errors="coerce")   # use the DataFrame directly
genes = ["PTDSS1", "PTDSS2"]
data = {}
for g in genes:
    sat, pufa = [], []
    for ann, v in df[f"{g}_log2FC"].items():
        db = ps_db(ann)
        if db is None or pd.isna(v):
            continue
        if db <= 1:
            sat.append(float(v))
        elif db >= 4:
            pufa.append(float(v))
        # DB 2 and 3 are excluded
    data[g] = (np.array(sat), np.array(pufa), mannwhitneyu(sat, pufa).pvalue)

n_sat = len(data[genes[0]][0])
n_pufa = len(data[genes[0]][1])

# plot
fig, ax = plt.subplots(figsize=(6.6, 6.2), dpi=150)
width, gap, centers = 0.34, 0.42, [1.0, 2.2]
rng = np.random.default_rng(0)
all_vals = []

for ci, g in enumerate(genes):
    sat, pufa, p = data[g]
    all_vals += sat.tolist() + pufa.tolist()
    posS, posP = centers[ci] - gap / 2, centers[ci] + gap / 2
    for pos, vals, fill, pt in [(posS, sat, GREEN_FILL, GREEN_PT),
                                (posP, pufa, PURPLE_FILL, PURPLE_PT)]:
        bp = ax.boxplot(vals, positions=[pos], widths=width, patch_artist=True,
                        showfliers=False, whis=1.5, zorder=1)
        for box in bp["boxes"]:
            box.set(facecolor=fill, edgecolor="black", linewidth=1)
        for el in ("whiskers", "caps"):
            for line in bp[el]:
                line.set(color="black", linewidth=1)
        for med in bp["medians"]:
            med.set(color="black", linewidth=2)
        jit = rng.uniform(-width * 0.42, width * 0.42, size=len(vals))
        ax.scatter(pos + jit, vals, s=30, facecolor=pt, edgecolor="black",
                   linewidth=0.5, alpha=0.85, zorder=2)
    ytop = max(sat.max(), pufa.max())
    ax.plot([posS, posS, posP, posP], [ytop + 0.10, ytop + 0.14, ytop + 0.14, ytop + 0.10],
            color="black", linewidth=1, zorder=3)
    ax.text((posS + posP) / 2, ytop + 0.16, f"{stars(p)}  p={p:.2g}",
            ha="center", va="bottom", fontsize=12)

ax.axhline(0, color="grey", linestyle="--", linewidth=1, zorder=0)
ax.set_xticks(centers)
ax.set_xticklabels(["PTDSS1 KO", "PTDSS2 KO"], fontsize=14, fontweight="bold")
ax.set_ylabel("log2 fold change vs WT", fontsize=14, fontweight="bold")
ax.set_title("PTDSS1 vs PTDSS2 KO — PS species by saturation", fontsize=14, fontweight="bold")
ax.tick_params(length=5, width=1)
ax.set_xlim(0.45, 2.75)

handles = [Patch(facecolor=GREEN_FILL, edgecolor="black", label=f"sat/MUFA-PS (DB\u22641) (n={n_sat})"),
           Patch(facecolor=PURPLE_FILL, edgecolor="black", label=f"PUFA-PS (DB\u22654) (n={n_pufa})")]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.10),
          frameon=False, fontsize=12, handlelength=1.4)

fig.tight_layout()
# fig.savefig("PTDSS1_PTDSS2_satMUFA_PUFA_boxplot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
"""Figure: PCYT2 knockout reroutes ether lipids from PE into PC.

For each ether species (and, as a specificity control, the median of the diacyl
PC and PE species) every one of the 143 gene knockouts is plotted as a grey dot;
PCYT2 is highlighted. Uses the shared publication style.
"""

from __future__ import annotations


from style import set_publication_style   # shared styling module

LIPID_CSV = all_results_df
OUT_PDF = "/staging/leuven/stg_00077/projects/jelle/lipogrid/pilot/analysis/final_4_runs/Figure_PCYT2_ether.pdf"

BLUE = "#1d5dab"     # highlight, matches the volcano blue used elsewhere
GREY = "#c9c9c9"     # the other 142 knockouts
RULE = "#8c8c8c"


def load_log2fc(source=LIPID_CSV) -> pd.DataFrame:
    """Return a lipid x gene matrix of log2 fold changes.

    `source` may be a path to the wide CSV, an already-loaded wide DataFrame
    (with *_log2FC / *_pvalue column pairs), or a DataFrame that is already a
    lipid x gene log2FC matrix.
    """
    if isinstance(source, (str, Path)):
        df = pd.read_csv(source, index_col=0)
    elif isinstance(source, pd.DataFrame):
        df = source
    else:
        raise TypeError(f"expected a path or DataFrame, got {type(source).__name__}")

    cols = [c for c in df.columns if str(c).endswith("_log2FC")]
    if not cols:                          # already a log2FC matrix
        return df.astype(float)
    fc = df[cols].astype(float)
    fc.columns = [str(c).rsplit("_", 1)[0] for c in cols]
    return fc


def lipid_class(name: str) -> str:
    m = re.match(r"^([A-Za-z0-9]+)\s", name)
    return m.group(1) if m else name


def build_rows(fc: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Three ether species, then the diacyl PC / PE class medians."""
    ether = [i for i in fc.index if " O-" in i or " P-" in i]
    ether = sorted(ether, key=lambda i: -fc.loc[i, "PCYT2"])
    rows = [(i, fc.loc[i]) for i in ether]
    for cls in ("PC", "PE"):
        idx = [i for i in fc.index
               if lipid_class(i) == cls and " O-" not in i and " P-" not in i]
        rows.append((f"{cls} diacyl (median of {len(idx)})", fc.loc[idx].median(axis=0)))
    return rows


def main() -> None:
    set_publication_style()
    fc = load_log2fc()
    rows = build_rows(fc)
    n_ether = sum(1 for lab, _ in rows if " O-" in lab or " P-" in lab)

    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    fig.subplots_adjust(left=0.30, right=0.975, top=0.90, bottom=0.16)

    # extra vertical gap between the ether block and the diacyl controls
    y_pos = [len(rows) - 1 - i - (0.0 if i < n_ether else 0.55)
             for i in range(len(rows))]

    for i, (label, values) in enumerate(rows):
        y = y_pos[i]
        others = values.drop("PCYT2")
        ax.scatter(others, y + rng.uniform(-0.16, 0.16, len(others)),
                   s=11, c=GREY, lw=0, zorder=1)

        value = values["PCYT2"]
        rank = int(values.rank(ascending=False)["PCYT2"])
        ax.scatter([value], [y], s=90, facecolor=BLUE, edgecolor="white",
                   lw=1.4, zorder=3)
        ax.annotate(f"{value:+.2f}   rank {rank}/{len(values)}".replace("-0.00", "0.00"),
                    (value, y), textcoords="offset points", xytext=(0, 14),
                    ha="center", va="bottom", fontsize=11, zorder=4)

    ax.axvline(0, color=RULE, lw=1, ls=(0, (4, 4)), zorder=0)
    ax.axhline(len(rows) - n_ether - 0.30, color=GREY, lw=1, zorder=0)

    ax.set_yticks(y_pos)
    ax.set_yticklabels([lab for lab, _ in rows], fontsize=12)
    ax.set_xlabel("log$_2$FC vs intergenic controls")
    ax.set_xlim(-1.5, 3.1)
    ax.set_ylim(-1.30, len(rows) - 0.15)
    ax.set_title("PCYT2 loss reroutes ether lipids into PC", loc="left", pad=10)
    ax.text(0.99, 0.02, "blue, PCYT2 knockout   ·   grey, other 142 knockouts",
            transform=ax.transAxes, ha="right", fontsize=10, color="#4d4d4d")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    fig.savefig(OUT_PDF)
    fig.savefig(OUT_PDF.replace(".pdf", ".png"), dpi=400)


if __name__ == "__main__":
    main()